## Poisson Distribution

The model is founded on the number of goals scored/conceded by each team. Teams that have been higher scorers in the past have a greater likelihood of scoring goals in the future. We'll import all match results from the recently concluded Premier League (2016/17) season. There's various sources for this data out there ([kaggle](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017?resource=download)).

In [ ]:
!git clone https://github.com/matthewli-melco/train-fb

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn
from scipy.stats import poisson,skellam
from scipy.stats import poisson

world_cup = pd.read_csv("/content/train-fb/filtered_results-edited.csv")
world_cup = world_cup[['HomeTeam','AwayTeam','FTHG','FTAG']]
world_cup = world_cup.rename(columns={'FTHG': 'HomeGoals', 'FTAG': 'AwayGoals'})
world_cup.head()

We imported a csv as a pandas dataframe, which contains various information for each of the >1000 world cup games in the 2020-2026 season. We restricted the dataframe to the columns in which we're interested (specifically, team names and numer of goals scored by each team).  Our task is to model the final round of fixtures in the season, so we must remove the last 10 rows (each gameweek consists of 10 matches).

In [ ]:
# Slices the DataFrame to remove the last 10 rows
world_cup = world_cup[:-10]

# FIX: Add numeric_only=True to prevent errors with string/categorical data
df_means = world_cup.mean(numeric_only=True)

print(df_means)

You'll notice that, on average, the home team scores more goals than the away team. This is the so called 'home (field) advantage'. This is a convenient time to introduce the [Poisson distribution](https://en.wikipedia.org/wiki/Poisson_distribution). It's a discrete probability distribution that describes the probability of the number of events within a specific time period (e.g 90 mins) with a known average rate of occurrence. A key assumption is that the number of events is independent of time. In our context, this means that goals don't become more/less likely by the number of goals already scored in the match. Instead, the number of goals is expressed purely as function an average rate of goals. If that was unclear, maybe this mathematical formulation will make clearer:

$$
P\left( x \right) = \frac{e^{-\lambda} \lambda ^x }{x!}, \lambda>0
$$

$\lambda$ represents the average rate (e.g. average number of goals, average number of letters you receive, etc.). So, we can treat the number of goals scored by the home and away team as two independent Poisson distributions. The plot below shows the proportion of goals scored compared to the number of goals estimated by the corresponding Poisson distributions.

# Turning a Poisson into a Machine Learning Representation


---

## Deriving the GLM Loss Function

Given your original equation for a single observation $x$ with an expected rate of occurrence $\lambda$:

$$P(x | \lambda) = \frac{e^{-\lambda} \lambda^x}{x!}$$


Transforming into the Natural Logarithm ($\ln$) Function:
$$\ln P(x | \lambda) = \ln\left(e^{-\lambda}\right) + \ln\left(\lambda^x\right) - \ln(x!)$$


Now, we get the simplified log-likelihood expression:
$$\ell(\lambda) = -\lambda + x \ln(\lambda) - \ln(x!)$$


Additionally, because the term $\ln(x!)$ depends purely on the actual data points ($x$) and does not contain the parameter $\lambda$ that we want to optimize, it acts as a constant. Dropping this constant term leaves us with the standard loss function.


In [ ]:
# Calculate means only for the specific numeric goals columns
means = world_cup[['HomeGoals', 'AwayGoals']].mean().values
poisson_pred = np.column_stack([[poisson.pmf(i, means[j]) for i in range(9)] for j in range(2)])

fig, ax = plt.subplots(figsize=(9, 4))

# Use specific bins (from -0.5 to 8.5) to perfectly center bars over whole numbers
plt.hist(world_cup[['HomeGoals', 'AwayGoals']].values, bins=np.arange(10)-0.5,
         alpha=0.7, label=['Home', 'Away'], density=True, color=["#FFA07A", "#20B2AA"])

# Plot Poisson lines directly over whole numbers (0 to 8) to align with histogram bins
pois1, = plt.plot(range(9), poisson_pred[:, 0],
                  linestyle='-', marker='o', label="Home", color='#CD5C5C')
pois2, = plt.plot(range(9), poisson_pred[:, 1],
                  linestyle='-', marker='o', label="Away", color='#006400')

leg = plt.legend(loc='upper right', fontsize=13, ncol=2)
leg.set_title("Poisson           Actual        ", prop={'size': '14', 'weight': 'bold'})

# Set positions and labels to match exactly (0 to 8)
plt.xticks(range(9))
plt.xlim(-0.7, 8.5)

plt.xlabel("Goals per Match", size=13)
plt.ylabel("Proportion of Matches", size=13)
plt.title("Number of Goals per Match (WC 2000/2026 Season)", size=14, fontweight='bold')
plt.ylim([-0.004, 0.4])
plt.tight_layout()

plt.show()

We can use this statistical model to estimate the probability of specfic events.

$$
\begin{align*}
P(\geq 2|Home) &= P(2|Home) + P(3|Home) + ...\\
        &= 0.22 + 0.11 + ...\\
        &= 0.33
\end{align*}
$$

The probability of a draw is simply the sum of the events where the two teams score the same amount of goals.

$$
\begin{align*}
P(Draw) &= P(0|Home) \times P(0|Away) + P(1|Home) \times P(1|Away) + ...\\
        &= 0.26 \times 0.39 + 0.3 \times 0.32 + ...\\
        &= 0.1974
\end{align*}
$$

Note that we consider the number of goals scored by each team to be independent events (i.e. P(A n B) = P(A) P(B)). The difference of two Poisson distribution is actually called a [Skellam distribution](https://en.wikipedia.org/wiki/Skellam_distribution). So we can calculate the probability of a draw by inputting the mean goal values into this distribution.

In [ ]:
# Calculate the mean by selecting the goal columns explicitly by name
means = world_cup[['HomeGoals', 'AwayGoals']].mean()

# pass the explicit column means into the Skellam PMF
# Skellam distribution models the difference between two independent Poisson distributions (Home - Away = 0 for a draw)
prob_draw = skellam.pmf(0, means['HomeGoals'], means['AwayGoals'])

print(f"Probability of a draw: {prob_draw:.4f}")

In [ ]:
# Calculate the mean by selecting the goal columns explicitly by name
means = world_cup[['HomeGoals', 'AwayGoals']].mean()

# Pass the explicit column means into the Skellam PMF
# x=1 calculates the exact probability of HomeGoals - AwayGoals = 1
prob_home_win_by_1 = skellam.pmf(1, means['HomeGoals'], means['AwayGoals'])

print(f"Probability of Home Team winning by exactly 1 goal: {prob_home_win_by_1:.4f}")

In [ ]:

# Calculate column means explicitly by name to avoid Pandas TypeErrors
home_mean = world_cup['HomeGoals'].mean()
away_mean = world_cup['AwayGoals'].mean()

# Define the integer range of goal differences to evaluate (-6 to 7)
goal_diff_range = range(-6, 8)

# Calculate Skellam probabilities
skellam_pred = [skellam.pmf(i, home_mean, away_mean) for i in goal_diff_range]

fig, ax = plt.subplots(figsize=(9, 4))

# Calculate the actual goal differences as a 1D Series
actual_diffs = world_cup['HomeGoals'] - world_cup['AwayGoals']

# Center the histogram bins exactly on the integers by shifting edges by -0.5
bins = np.arange(-6, 9) - 0.5
plt.hist(actual_diffs, bins=bins, alpha=0.7, label='Actual', density=True, color='#20B2AA')

# Plot Skellam directly on the integer coordinates (no i+0.5 shift needed)
plt.plot(list(goal_diff_range), skellam_pred, linestyle='-', marker='o', label="Skellam", color='#CD5C5C')

# Formatting updates
plt.legend(loc='upper right', fontsize=13)
plt.xticks(list(goal_diff_range))  # Ticks line up perfectly with integers now
plt.xlabel("Home Goals - Away Goals", size=13)
plt.ylabel("Proportion of Matches", size=13)
plt.title("Difference in Goals Scored (Home Team vs Away Team)", size=14, fontweight='bold')
plt.ylim([-0.004, 0.26])
plt.tight_layout()

plt.show()

So, hopefully you can see how we can adapt this approach to model specific matches. We just need to know the average number of goals scored by each team and feed this data into a Poisson model. Let's have a look at the distribution of goals scored by Gersea and Chiderland (teams who finished 1st and last, respectively).

In [ ]:
# Set up the subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# 1. FIX: Use single brackets to get clean 1D Series for value_counts
Ger_home = world_cup[world_cup['HomeTeam']=='Germany']['HomeGoals'].value_counts(normalize=True)
Chi_home = world_cup[world_cup['HomeTeam']=='Chile']['HomeGoals'].value_counts(normalize=True)
Ger_away = world_cup[world_cup['AwayTeam']=='Germany']['AwayGoals'].value_counts(normalize=True)
Chi_away = world_cup[world_cup['AwayTeam']=='Chile']['AwayGoals'].value_counts(normalize=True)

# 2. FIX: Simplify lambda mean goals calculations using native .mean()
Ger_home_pois = [poisson.pmf(i, world_cup[world_cup['HomeTeam']=='Germany']['HomeGoals'].mean()) for i in range(8)]
Chi_home_pois = [poisson.pmf(i, world_cup[world_cup['HomeTeam']=='Chile']['HomeGoals'].mean()) for i in range(8)]
Ger_away_pois = [poisson.pmf(i, world_cup[world_cup['AwayTeam']=='Germany']['AwayGoals'].mean()) for i in range(8)]
Chi_away_pois = [poisson.pmf(i, world_cup[world_cup['AwayTeam']=='Chile']['AwayGoals'].mean()) for i in range(8)]

# --- Plot 1: HOME ---
# Adjusted widths to (-0.2 and +0.2) so bars sit perfectly symmetrical side-by-side
ax1.bar(Ger_home.index - 0.2, Ger_home.values, width=0.4, color="#034694", label="Germany")
ax1.bar(Chi_home.index + 0.2, Chi_home.values, width=0.4, color="#EB172B", label="Chile")

ax1.plot(range(8), Ger_home_pois, linestyle='-', marker='o', label="Germany (Poisson)", color="#0a7bff")
ax1.plot(range(8), Chi_home_pois, linestyle='-', marker='o', label="Chile (Poisson)", color="#ff7c89")

leg = ax1.legend(loc='upper right', fontsize=11, ncol=2)
leg.set_title("Poisson                 Actual                ", prop={'size': '12', 'weight': 'bold'})
ax1.set_xlim([-0.5, 7.5])
ax1.set_ylim([-0.01, 0.65])

# Replaced deprecated set_xticklabels([]) with modern tick_params to hide top labels safely
ax1.tick_params(labelbottom=False)

# --- Plot 2: AWAY ---
ax2.bar(Ger_away.index - 0.2, Ger_away.values, width=0.4, color="#034694", label="Germany")
ax2.bar(Chi_away.index + 0.2, Chi_away.values, width=0.4, color="#EB172B", label="Chile")

ax2.plot(range(8), Ger_away_pois, linestyle='-', marker='o', label="Germany (Poisson)", color="#0a7bff")
ax2.plot(range(8), Chi_away_pois, linestyle='-', marker='o', label="Chile (Poisson)", color="#ff7c89")
ax2.set_xlim([-0.5, 7.5])
ax2.set_ylim([-0.01, 0.65])

# Facet Labels (Ggplot2 style side-banners)
# Adjusted the Y position to 0.3 so they sit centered in the subplots
ax1.text(7.65, 0.3, '                Home                ', rotation=-90,
         bbox={'facecolor': '#ffbcf6', 'alpha': 0.5, 'pad': 5}, va='center')
ax2.text(7.65, 0.3, '                Away                ', rotation=-90,
         bbox={'facecolor': '#ffbcf6', 'alpha': 0.5, 'pad': 5}, va='center')

# Axis Titles and Overall Labels
ax1.set_title("Number of Goals per Match (EWC 2000/2026 Season)", size=14, fontweight='bold')
ax2.set_xlabel("Goals per Match", size=13)

# Replaced unstable manual text positioning with a clean global figure label
fig.text(0.01, 0.5, 'Proportion of Matches', rotation=90, size=13, va='center', ha='center')

plt.tight_layout()
plt.show()

## Building A Model

You should now be convinced that the number of goals scored by each team can be approximated by a Poisson distribution. Due to a relatively sample size, the accuracy of this approximation can vary significantly (especially earlier in the season when teams have played fewer games). Similar to before, we could now calculate the probability of various events in this World Cup match. But rather than treat each match separately, we'll build a more general Poisson regression model ([what is that?](https://en.wikipedia.org/wiki/Poisson_regression)).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Clean the initial dataset
world_cup['HomeGoals'] = pd.to_numeric(world_cup['HomeGoals'], errors='coerce')
world_cup['AwayGoals'] = pd.to_numeric(world_cup['AwayGoals'], errors='coerce')
world_cup = world_cup.dropna(subset=['HomeGoals', 'AwayGoals', 'HomeTeam', 'AwayTeam'])

# 2. Reshape the DataFrame
goal_model_data = pd.concat([
    world_cup[['HomeTeam', 'AwayTeam', 'HomeGoals']].assign(home=1).rename(
        columns={'HomeTeam': 'team', 'AwayTeam': 'opponent', 'HomeGoals': 'goals'}),
    world_cup[['AwayTeam', 'HomeTeam', 'AwayGoals']].assign(home=0).rename(
        columns={'AwayTeam': 'team', 'HomeTeam': 'opponent', 'AwayGoals': 'goals'})
])

# 3. CRITICAL FIX A: Ensure goals are integers and drop any remaining NaN/Inf values
goal_model_data['goals'] = goal_model_data['goals'].astype(int)
goal_model_data = goal_model_data[np.isfinite(goal_model_data['goals'])]

# 4. CRITICAL FIX B: Drop teams with too few matches (prevents perfect separation)
# Teams with 1 or 2 matches can easily cause the exponent to overflow
team_counts = goal_model_data['team'].value_counts()
valid_teams = team_counts[team_counts >= 3].index
goal_model_data = goal_model_data[goal_model_data['team'].isin(valid_teams) & goal_model_data['opponent'].isin(valid_teams)]

# Define the categorical formula
formula = "goals ~ home + C(team) + C(opponent)"

# Use a robust optimization technique to handle convergence safely
try:
    # Attempt regular fitting
    poisson_model = smf.glm(formula=formula, data=goal_model_data, family=sm.families.Poisson()).fit()
except ValueError:
    print("Standard solver failed due to mathematical instability. Switching to regularized fit...")
    # fit_regularized applies a tiny penalty to prevent coefficients from sliding to infinity
    poisson_model = smf.glm(formula=formula, data=goal_model_data, family=sm.families.Poisson()).fit_regularized(alpha=1e-4)

# Print the summary results
print(poisson_model.summary())

In [ ]:
poisson_model.predict(pd.DataFrame(data={'team': 'Germany', 'opponent': 'Chile',
                                       'home':1},index=[1]))

In [ ]:
poisson_model.predict(pd.DataFrame(data={'team': 'Chile', 'opponent': 'Germany',
                                       'home':0},index=[1]))

Just like before, we have two Poisson distributions. From this, we can calculate the probability of various events. I'll wrap this in a `simulate_match` function.

In [ ]:
def simulate_match(foot_model, homeTeam, awayTeam, max_goals=10):
    home_goals_avg = foot_model.predict(pd.DataFrame(data={'team': homeTeam,
                                                            'opponent': awayTeam,'home':1},
                                                      index=[1])).values[0]
    away_goals_avg = foot_model.predict(pd.DataFrame(data={'team': awayTeam,
                                                            'opponent': homeTeam,'home':0},
                                                      index=[1])).values[0]
    team_pred = [[poisson.pmf(i, team_avg) for i in range(0, max_goals+1)] for team_avg in [home_goals_avg, away_goals_avg]]
    return(np.outer(np.array(team_pred[0]), np.array(team_pred[1])))
simulate_match(poisson_model, 'Germany', 'Chile', max_goals=3)

This matrix simply shows the probability of Germany (rows of the matrix) and Chilie (matrix columns) scoring a specific number of goals. For example, along the diagonal, both teams score the same the number of goals (e.g. P(0-0)=0.031). So, you can calculate the odds of draw by summing all the diagonal entries. Everything below the diagonal represents a Gersea victory (e.g P(3-0)=0.149), And you can estimate P(Over 2.5 goals) by summing all entries except the four values in the upper left corner. Luckily, we can use basic matrix manipulation functions to perform these calculations.

In [ ]:
# Germany win
Ger_Chi = simulate_match(poisson_model, "Germany", "Chile", max_goals=10)
np.sum(np.tril(Ger_Chi, -1))

In [ ]:
# draw
np.sum(np.diag(Ger_Chi))

In [ ]:
# Chile win
np.sum(np.triu(Ger_Chi, 1))